<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/MinhKhoa/test_obesity_lassovsboruta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LassoCV

In [8]:
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/conquer1/AIO-Conquer-Data/Data/dataset/obesity_encoded.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tải dữ liệu thành công! Kích thước: (2087, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 33 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 2087 non-null   float64
 1   Height                              2087 non-null   float64
 2   Weight                              2087 non-null   float64
 3   FCVC                                2087 non-null   float64
 4   NCP                                 2087 non-null   float64
 5   CH2O                                2087 non-null   float64
 6   FAF                                 2087 non-null   float64
 7   TUE                                 2087 non-null   float64
 8   NObeyesdad                          2087 non-null   int64  

In [9]:
target_colum = 'Weight'
X = df.drop(columns=[target_colum,'BMI'], errors='ignore')
y = df[target_colum]

# 2. Chia tập Train/Test trước để tránh data leakage khi tính trung bình giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# 1. Khởi tạo mô hình hồi quy Gradient Boosting
gbr = GradientBoostingRegressor(max_depth=5, random_state=42)

# 2. Huấn luyện mô hình trên tập dữ liệu đã encode
gbr.fit(X_train, y_train)

# 3. Tiến hành dự đoán giá nhà
preds = gbr.predict(X_test)

# 4. Đánh giá mô hình bằng R2-score (Giá trị càng gần 1 hay 100% thì mô hình càng tốt)
r2_score_all = round(r2_score(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")

Chỉ số R2-score khi sử dụng tất cả feature: 0.987


# **1.Boruta**

In [11]:
# --- Bước 1: Lấy danh sách tên thuộc tính ban đầu ---
feature_names = df.drop(columns=[target_colum], errors='ignore').columns.tolist()

# --- Bước 2: Chuyển đổi dữ liệu thành mảng Numpy để tránh lỗi Index ---
X_train_np = X_train.values if hasattr(X_train, 'columns') else X_train
X_test_np = X_test.values if hasattr(X_test, 'columns') else X_test
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# --- Bước 3: Khởi tạo mô hình hồi quy lõi ---
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

# --- Bước 4: Cấu hình và chạy Boruta (Ẩn nhật ký chạy) ---
boruta_selector = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    verbose=0,
    random_state=42,
    max_iter=100
)
boruta_selector.fit(X_train_np, y_train_np)

# --- Bước 5: Lọc dữ liệu theo các feature được chọn ---
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test_np)

# --- Bước 6: Huấn luyện lại mô hình trên tập dữ liệu đã lọc để tính R2-score ---
# Cách này giúp tránh hoàn toàn lỗi gọi thuộc tính 'estimator_' của thư viện Boruta
rf_final = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
rf_final.fit(X_train_filtered, y_train_np)

# Tiến hành dự đoán và tính điểm R2
preds_boruta = rf_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test_np, preds_boruta), 3)

# Lọc danh sách thuộc tính được chọn chính thức
confirmed_features = [feature_names[i] for i, confirmed in enumerate(boruta_selector.support_) if confirmed]

# --- Bước 7: In kết quả ngắn gọn theo yêu cầu ---
print("="*60)
print(f"Chỉ số R2-score khi sử dụng Boruta: {r2_score_boruta}")
print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")
print(f"Số lượng tính năng được giữ lại: {len(confirmed_features)} / {len(feature_names)}")
print(f"Các tính năng được chọn: {confirmed_features}")
print("="*60)

Chỉ số R2-score khi sử dụng Boruta: 0.979
Chỉ số R2-score khi sử dụng tất cả feature: 0.987
Số lượng tính năng được giữ lại: 14 / 32
Các tính năng được chọn: ['Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'NObeyesdad', 'BMI', 'Gender_Female', 'CALC_Frequently', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Motorbike']


# **2.LASSO**

In [12]:
# --- Bước 1: Chuẩn hóa dữ liệu (Feature Scaling) ---
# Lasso cần dữ liệu cùng thang đo để phạt (penalty) các hệ số một cách công bằng
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bước 2: Khởi tạo và huấn luyện LassoCV ---
# Sử dụng LassoCV để tự động tìm tham số alpha (hệ số phạt) tối ưu nhất thông qua Cross-Validation
lasso = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, random_state=42, n_jobs=-1)
lasso.fit(X_train_scaled, y_train)

# --- Bước 3: Dự đoán và tính toán điểm R2-score ---
preds_lasso = lasso.predict(X_test_scaled)
r2_score_lasso = round(r2_score(y_test, preds_lasso), 3)

# --- Bước 4: Thống kê các feature bị Lasso loại bỏ (hệ số hệ số bằng 0) ---
feature_names = X.columns if hasattr(X, 'columns') else [f"Feature_{i}" for i in range(X.shape[1])]
coefficients = lasso.coef_

# Lọc ra các biến được giữ lại và các biến bị triệt tiêu về 0
kept_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef != 0]
dropped_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef == 0]

# --- Bước 5: In kết quả so sánh với Baseline ---
print("="*60)
print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")
print(f"[+] Chỉ số R2-score khi sử dụng Lasso: {r2_score_lasso}")
print(f" Alpha tối ưu được chọn: {round(lasso.alpha_, 5)}")
print("="*60)
print(f" Số lượng tính năng bị Lasso loại bỏ (Hệ số = 0): {len(dropped_features)} / {len(feature_names)}")
if len(dropped_features) > 0:
    print(f"   Các tính năng bị loại bỏ: {dropped_features}")
print(f"\n Số lượng tính năng được giữ lại: {len(kept_features)}")
print(f"   Các tính năng được giữ lại: {kept_features}")
print("="*60)

Chỉ số R2-score khi sử dụng tất cả feature: 0.987
[+] Chỉ số R2-score khi sử dụng Lasso: 0.96
 Alpha tối ưu được chọn: 0.03352
 Số lượng tính năng bị Lasso loại bỏ (Hệ số = 0): 4 / 31
   Các tính năng bị loại bỏ: ['Gender_Male', 'CAEC_Always', 'CALC_Frequently', 'MTRANS_Walking']

 Số lượng tính năng được giữ lại: 27
   Các tính năng được giữ lại: ['Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'NObeyesdad', 'Gender_Female', 'family_history_with_overweight_no', 'family_history_with_overweight_yes', 'FAVC_no', 'FAVC_yes', 'CAEC_Frequently', 'CAEC_Sometimes', 'CAEC_no', 'SMOKE_no', 'SMOKE_yes', 'SCC_no', 'SCC_yes', 'CALC_Always', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Automobile', 'MTRANS_Bike', 'MTRANS_Motorbike', 'MTRANS_Public_Transportation']
